# Week 3: Model Prototyping, Imbalance Strategy & PR-AUC Evaluation

This notebook walks through the machine learning model training process for our real-time fraud detection system.

### Key Technical Takeaways for Interviews:
1. **Metric Selection**: Why Accuracy and ROC-AUC are insufficient for severe class imbalance (~0.13% fraud), and why **PR-AUC (Average Precision)** is the primary optimization target.
2. **Cost-Sensitive Learning**: How `scale_pos_weight` penalizes false negatives (missed frauds) without requiring distortive synthetic oversampling (SMOTE).
3. **Threshold Tuning on Validation**: Why the default 0.5 decision threshold is arbitrary and how sweeping the PR curve yields an optimal decision threshold $\tau^*$.
4. **Tri-Tier Decision Engine**: Operationalizing risk scores into `AUTO_APPROVE`, `MANUAL_REVIEW`, and `AUTO_BLOCK`.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
from src.config import MODELS_DIR, PROCESSED_DATA_DIR
from src.features import FEATURE_NAMES

# 1. Load evaluation metrics from production run
with open(MODELS_DIR / 'eval_metrics.json') as f:
    metrics = json.load(f)

print("Champion Model:", metrics['champion_model'])
print("Tuned Threshold (tau*):", metrics['tuning_policy']['best_threshold'])
print("Three-Tier Policy:", f"Allow < {metrics['tuning_policy']['tau_allow']} | Review [{metrics['tuning_policy']['tau_allow']}, {metrics['tuning_policy']['tau_block']}) | Block >= {metrics['tuning_policy']['tau_block']}")

## 1. Model Comparison on Out-of-Time Test Set

We compare three distinct models on the chronologically partitioned Test set (`steps 651–743`):

In [ ]:
comparison_df = pd.DataFrame.from_dict(metrics['models_comparison'], orient='index')
comparison_df[['split', 'pr_auc', 'roc_auc', 'precision', 'recall', 'f1_score']]

## 2. Confusion Matrix Analysis (Champion Model)

Evaluating false positive vs false negative financial trade-offs on the 80,831 test transactions.

In [ ]:
cm = metrics['models_comparison']['xgboost']['confusion_matrix']
print(f"True Negatives  (Legitimate Allowed) : {cm['true_negatives']:,}")
print(f"False Positives (Legitimate Flagged) : {cm['false_positives']:,}")
print(f"False Negatives (Frauds Missed)     : {cm['false_negatives']:,}")
print(f"True Positives  (Frauds Caught)      : {cm['true_positives']:,}")

## 3. Feature Importance Ranking

Understanding which behavioral signals drive the model's fraud predictions.

In [ ]:
imp_series = pd.Series(metrics['feature_importances']).sort_values(ascending=False)
print("Top 10 Most Important Features:")
imp_series.head(10)